# Assignment 1: Introduction to Language Modeling

RNN-based autoregressive language model on Wikipedia paragraphs.
Implementations of `A1Tokenizer`, `A1RNNModel`, and `A1Trainer` live in `tokenizer.py`, `model.py`, `trainer.py`.

## Setup (run on Colab)

Mount Drive (optional, only if you want to persist the trained model across sessions), clone the repo, install deps, download data.

In [ ]:
# If your tokenizer.py / model.py / trainer.py are in your GitHub repo,
# clone it and cd into the assignment folder. Adjust the path as needed.
# (Skip this cell if you uploaded the .py files manually to the Colab session.)

# !git clone https://github.com/dmw1998/WASP_DL4NLP26.git
# %cd WASP_DL4NLP26/assignments/a1_1

In [ ]:
# Download and extract the assignment data (text files + the skeleton).
# Skip if you already have wiki.train.txt / wiki.valid.txt in the current dir.
!wget -q https://www.cse.chalmers.se/~richajo/waspnlp2026/a1_1.zip
!unzip -oq a1_1.zip
!ls

In [ ]:
%load_ext autoreload
%autoreload 2

import os, math
import torch
import nltk

# Punkt tokenizer data is needed by NLTK's word_tokenize.
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# !! Check the actual filenames after unzipping and adjust if needed.
TRAIN_FILE = 'a1_1/train.txt'
VAL_FILE = 'a1_1/valid.txt'

## Part 1: Tokenization

Build the tokenizer directly from the training file. The default `tokenize_fun` (`lowercase_tokenizer`) does NLTK word-splitting + lowercase. The 4 special tokens (`<PAD>`, `<UNK>`, `<BOS>`, `<EOS>`) occupy the first 4 ids.

In [ ]:
from tokenizer import build_tokenizer, A1Tokenizer, lowercase_tokenizer

MAX_VOC_SIZE = 10000
MODEL_MAX_LENGTH = 128

tokenizer = build_tokenizer(
    train_file=TRAIN_FILE,
    tokenize_fun=lowercase_tokenizer,
    max_voc_size=MAX_VOC_SIZE,
    model_max_length=MODEL_MAX_LENGTH,
)

print('vocab size:', len(tokenizer))
print('pad_token_id:', tokenizer.pad_token_id)
print('first 10 entries:', list(tokenizer.str_to_int.items())[:10])

In [ ]:
# Sanity checks (Task 1.2)
assert len(tokenizer) <= MAX_VOC_SIZE
for tok in ['<PAD>', '<UNK>', '<BOS>', '<EOS>']:
    assert tok in tokenizer.str_to_int, f'{tok} missing'
for w in ['the', 'and', 'of']:
    assert w in tokenizer.str_to_int, f'common word {w} should be in vocab'
for w in ['cuboidal', 'epiglottis']:
    if w not in tokenizer.str_to_int:
        print(f'OK: rare word {w!r} not in vocab')

# Round-trip check.
i = tokenizer.str_to_int['the']
assert tokenizer.int_to_str[i] == 'the'
print('sanity checks passed')

In [ ]:
# Task 1.3 sanity check: __call__ on inputs of different lengths.
test_texts = ['This is a test.', 'Another test.']
enc = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
print(enc)
print('decoded[0]:', tokenizer.decode(enc['input_ids'][0]))
print('decoded[1]:', tokenizer.decode(enc['input_ids'][1]))

In [ ]:
# Save (and reload) the tokenizer so we don't rebuild it next session.
tokenizer.save('a1_tokenizer.pkl')
reloaded = A1Tokenizer.from_file('a1_tokenizer.pkl')
assert len(reloaded) == len(tokenizer)
print('save/load OK')

## Part 2: load datasets, quick batching demo

(Not required in the submission. Useful for sanity-checking the data pipeline.)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'text',
    data_files={'train': TRAIN_FILE, 'val': VAL_FILE},
)
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))
print('example:', dataset['train'][8]['text'][:200])

## Part 3: define the model

`A1RNNModel` follows the skeleton: embedding -> LSTM -> unembedding, returning a `CausalLMOutput`.

In [ ]:
from model import A1RNNModelConfig, A1RNNModel

config = A1RNNModelConfig(
    vocab_size=len(tokenizer),
    embedding_size=128,
    hidden_size=256,
    num_layers=1,
)
model = A1RNNModel(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'model parameters: {n_params:,}')
print(model)

In [ ]:
# Task 3 sanity check: 1 x N input -> 1 x N x V logits.
N = 7
dummy = torch.randint(0, len(tokenizer), (1, N))
with torch.no_grad():
    out = model(dummy)
print('logits shape:', out.logits.shape)
assert out.logits.shape == (1, N, len(tokenizer))

out = model(dummy, labels=dummy)
print('loss with labels:', out.loss.item())

## Part 4: train

Use HuggingFace `TrainingArguments`. Required by the skeleton: `optim='adamw_torch'`, `eval_strategy='epoch'`.

Set `dev_mode = True` to debug on 1000 examples (~1 minute on GPU).

In [ ]:
from torch.utils.data import Subset
from transformers import TrainingArguments
from trainer import A1Trainer

dev_mode = False

if dev_mode:
    train_ds = Subset(dataset['train'], range(1000))
    val_ds = Subset(dataset['val'], range(200))
    epochs = 1
else:
    train_ds = dataset['train']
    val_ds = dataset['val']
    epochs = 3

args = TrainingArguments(
    output_dir='trainer_output',
    optim='adamw_torch',
    eval_strategy='epoch',
    learning_rate=1e-3,
    num_train_epochs=epochs,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    logging_steps=100,
    save_strategy='no',           # we save manually at the end of training
    report_to='none',
    use_cpu=False,
)

trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()

## Part 5: evaluation

### Task 5.1: predict the next word

Encode a prompt, take the logits one position *before* `<EOS>`, read off the top-k.

In [ ]:
def predict_next(model, tokenizer, prompt, k=5):
    device = next(model.parameters()).device
    model.eval()
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    with torch.no_grad():
        out = model(input_ids)
    # Our tokenizer appends <EOS>, so the prediction for what follows the
    # last real word lives at position -2 (one before <EOS>).
    logits = out.logits[0, -2]
    topk = torch.topk(logits, k)
    return [
        (tokenizer.int_to_str[i.item()], s.item())
        for i, s in zip(topk.indices, topk.values)
    ]

for prompt in [
    'She lives in San',
    'The president of the United',
    'He played the guitar and',
]:
    print(f'prompt: {prompt!r}')
    for word, score in predict_next(model, tokenizer, prompt):
        print(f'  {word:20s} {score:.3f}')
    print()

### Task 5.2: perplexity on validation

In [ ]:
val_loss, val_ppl = trainer.evaluate()
print(f'validation cross-entropy: {val_loss:.4f}')
print(f'validation perplexity:    {val_ppl:.2f}')

### Task 5.3: inspect learned word embeddings

In [ ]:
from torch import nn

def nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5):
    if word not in voc:
        print(f'{word!r} not in vocab')
        return []
    test_emb = emb.weight[voc[word]]
    sim = nn.CosineSimilarity(dim=1)(test_emb, emb.weight)
    top = sim.topk(n_neighbors + 1)
    # Skip position 0: that's the query word itself.
    return [
        (inv_voc[ix.item()], cos.item())
        for ix, cos in zip(top.indices[1:], top.values[1:])
    ]

# Move embedding to CPU so cosine sim doesn't need GPU.
emb = model.embedding.cpu()
for w in ['sweden', 'king', 'three', 'london', 'small']:
    print(f'neighbors of {w!r}:')
    for nbr, score in nearest_neighbors(
        emb, tokenizer.str_to_int, tokenizer.int_to_str, w
    ):
        print(f'  {nbr:20s} {score:.3f}')
    print()

In [ ]:
# Optional: PCA scatterplot.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

def plot_embeddings_pca(emb, voc, words):
    words = [w for w in words if w in voc]
    vecs = np.vstack([emb.weight[voc[w]].detach().cpu().numpy() for w in words])
    vecs -= vecs.mean(axis=0)
    twodim = TruncatedSVD(n_components=2).fit_transform(vecs)
    plt.figure(figsize=(6, 6))
    plt.scatter(twodim[:, 0], twodim[:, 1], c='r', edgecolors='k')
    for w, (x, y) in zip(words, twodim):
        plt.text(x + 0.02, y, w)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_embeddings_pca(
    emb, tokenizer.str_to_int,
    ['sweden', 'denmark', 'europe', 'africa', 'london', 'stockholm',
     'large', 'small', 'great', 'black',
     'three', 'seven', 'ten'],
)

## Reloading later

```python
from model import A1RNNModel
from tokenizer import A1Tokenizer

tokenizer = A1Tokenizer.from_file('a1_tokenizer.pkl')
model = A1RNNModel.from_pretrained('trainer_output')
```